# 🛰️ Taller: del píxel al objeto — Segmentación de imágenes satelitales **en tu navegador**

**Curso de análisis de imágenes satelitales · Dr. Abel Coronado**

---

## 🧭 Antes de empezar: ¿dónde estás parado?

Esto que ves es **JupyterLab**, el entorno de trabajo estándar de la ciencia de datos. Pero esta versión tiene algo especial: **no está instalada en tu computadora ni corre en un servidor**. Todo el laboratorio — Python, las bibliotecas científicas y geoespaciales, tus datos — vive **dentro de esta pestaña del navegador**, gracias a una tecnología llamada **WebAssembly**.

¿Qué significa eso para ti?

| | |
|---|---|
| 🚫 **Nada que instalar** | No necesitas permisos de administrador ni descargar programas |
| 🔒 **Tus datos no viajan** | El procesamiento ocurre en TU equipo; nada se sube a ninguna nube |
| 🧹 **Cero rastro** | Cierras la pestaña y no queda nada instalado |
| ⚡ **Es Python de verdad** | El mismo código que usarías en un servidor profesional |

**Cómo se usa:** cada bloque gris es una *celda* de código. Haz clic en ella y presiona **Shift + Enter** para ejecutarla. Ve en orden, de arriba hacia abajo. La primera celda tarda unos segundos extra (carga las bibliotecas científicas la primera vez) — es normal.

Empecemos por comprobar que no te estoy mintiendo: 👇

In [ ]:
# ¿Dónde está corriendo este Python? Vamos a preguntárselo directamente.
import sys
import platform

print(f"Versión de Python : {sys.version.split()[0]}")
print(f"Sistema operativo : {sys.platform!r}")
print(f"Arquitectura      : {platform.machine()!r}")
print()
if sys.platform == "emscripten":
    print("✅ 'emscripten' y 'wasm32' significan: este Python corre en WebAssembly,")
    print("   DENTRO de tu navegador. No hay servidor. El laboratorio eres tú. 🚀")
else:
    print("ℹ️ Estás corriendo este cuaderno fuera del navegador (modo local).")

---
## 📚 Teoría 1: la imagen con la que vamos a trabajar

Trabajaremos con un recorte real de la **geomediana Sentinel-2 del año 2020 de Aguascalientes**.

- **Sentinel-2** es una misión de satélites del programa europeo Copernicus que fotografía toda la Tierra cada ~5 días, con píxeles de 10–20 metros. Sus imágenes son **públicas y gratuitas**.
- No mide solo el rojo, verde y azul que ven tus ojos: registra **12 bandas espectrales**, incluyendo infrarrojo cercano (NIR) e infrarrojo de onda corta (SWIR), donde la vegetación, el agua, el suelo y el concreto se distinguen muchísimo mejor.
- Una **geomediana** es el resultado de tomar *todas* las imágenes de un año y calcular, píxel por píxel, un valor mediano robusto. El resultado: una imagen **sin nubes ni sombras**, representativa de todo el año.

Así que cada píxel de nuestra imagen no es un color: es un **vector de 12 números** que describe cómo refleja la luz ese pedacito de 10×10 metros de Aguascalientes. Vamos a verla:

In [ ]:
import time
import numpy as np
import rasterio
from rasterio import features
import matplotlib.pyplot as plt
# Estos dos imports también PRE-CARGAN scipy y scikit-learn en el navegador:
# Pyodide instala paquetes al verlos importados en una celda, pero no detecta
# los imports internos de un módulo (como nuestro segmentador shepherd_pure).
import scipy.ndimage
import sklearn.cluster

with rasterio.open("tile_ags_256.tif") as src:
    img = src.read().astype(np.float32)   # (bandas, filas, columnas)
    transform, crs, nodata = src.transform, src.crs, src.nodata

n_bandas, alto, ancho = img.shape
print(f"Imagen: {alto}×{ancho} píxeles × {n_bandas} bandas  (≈ {alto*10/1000:.1f} km por lado)")
print(f"Sistema de coordenadas: proyección cónica de área igual para México (Albers)")

# Composición en color natural: bandas 4 (rojo), 3 (verde), 2 (azul)
rgb = np.stack([img[3], img[2], img[1]], axis=-1)
p2, p98 = np.percentile(rgb, (2, 98))     # estiramos el contraste entre percentiles
rgb = np.clip((rgb - p2) / (p98 - p2), 0, 1)

plt.figure(figsize=(6, 6))
plt.imshow(rgb)
plt.title("Geomediana Sentinel-2 2020 — recorte de Aguascalientes (color natural)")
plt.axis("off")
plt.show()
print("💡 Cada píxel que ves guarda 12 mediciones, no 3. Vemos solo 3 porque así son nuestros ojos.")

---
## 📚 Teoría 2: ¿por qué "segmentar"? Del píxel al objeto

Si quisiéramos clasificar esta imagen píxel por píxel ("¿este píxel es urbano o agrícola?"), tendríamos dos problemas: **ruido** (píxeles aislados mal clasificados, efecto "sal y pimienta") y **falta de contexto** (un píxel gris puede ser una calle, un techo o suelo desnudo — solo, no se sabe).

La alternativa es el enfoque **orientado a objetos (GEOBIA)**: primero agrupamos los píxeles en **segmentos** — regiones contiguas espectralmente homogéneas que corresponden a *cosas* del territorio: una parcela, una manzana, un cuerpo de agua. Después clasificamos los segmentos, no los píxeles.

Usaremos el algoritmo de **Shepherd, Bunting y Dymond (2019)** (*Remote Sensing* 11(6):658), el mismo que usan agencias de monitoreo territorial. Funciona en 3 pasos:

1. **Siembra (K-means):** agrupa los píxeles en ~60 "familias espectrales" según sus 12 bandas — sin importar dónde están.
2. **Aglomerado (clumping):** los píxeles *vecinos* que cayeron en la misma familia se unen en grupos contiguos. Salen miles de grupitos.
3. **Eliminación iterativa:** los grupos demasiado chicos (menos de `minSegmentSize` píxeles) se fusionan, del más pequeño al más grande, con el vecino espectralmente más parecido. Quedan solo objetos de tamaño razonable.

La implementación que vas a ejecutar (`shepherd_pure`) está **validada bit a bit** contra la implementación de referencia (`pyshepseg`): produce exactamente los mismos segmentos.

Veamos el **paso 1** con nuestros propios ojos:

In [ ]:
# PASO 1 — Siembra: K-means agrupa los píxeles en 60 familias espectrales
import shepherd_pure

t0 = time.time()
km = shepherd_pure.fitSpectralClusters(img, numClusters=60, subsamplePcnt=1,
                                       imgNullVal=nodata, fixedKMeansInit=True)
clusters = shepherd_pure.applySpectralClusters(km, img, nodata)
print(f"K-means listo en {time.time()-t0:.1f} s — cada píxel tiene ahora una 'familia' (1 a 60)")

plt.figure(figsize=(12, 5.5))
plt.subplot(1, 2, 1); plt.imshow(rgb); plt.title("La imagen"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(clusters, cmap="tab20", interpolation="nearest")
plt.title("Paso 1: familias espectrales (colores = familias)"); plt.axis("off")
plt.tight_layout(); plt.show()
print("💡 Observa: las familias capturan tipos de cobertura, pero quedan 'salpicadas'.")
print("   Los pasos 2 y 3 convierten esta sal y pimienta en objetos limpios.")

In [ ]:
# PASOS 2 y 3 — Aglomerar y depurar: la segmentación completa
# (reutilizamos el K-means que ya ajustamos: kmeansObj=km)
t0 = time.time()
res = shepherd_pure.doShepherdSegmentation(
    img,
    kmeansObj=km,          # paso 1 ya hecho
    minSegmentSize=50,     # tamaño mínimo de objeto: 50 px = media hectárea
    imgNullVal=nodata)
seg = res.segimg

print(f"Segmentación completa en {time.time()-t0:.1f} s — ¡dentro de tu navegador!")
print(f"  · objetos finales              : {int(seg.max()):,}")
print(f"  · píxeles sueltos absorbidos   : {res.singlePixelsEliminated:,}")
print(f"  · grupitos chicos fusionados   : {res.smallSegmentsEliminated:,}")
print(f"  · umbral espectral de fusión   : {res.maxSpectralDiff:.0f} (calculado automáticamente)")

In [ ]:
# Visualicemos el resultado: las fronteras de los objetos sobre la imagen
from scipy import ndimage

bordes = ndimage.maximum_filter(seg, size=2) != ndimage.minimum_filter(seg, size=2)
vis = rgb.copy()
vis[bordes] = [1, 1, 0]   # fronteras en amarillo

plt.figure(figsize=(7.5, 7.5))
plt.imshow(vis)
plt.title(f"Paso 2+3: {int(seg.max()):,} objetos del territorio (fronteras en amarillo)")
plt.axis("off")
plt.show()
print("💡 Ya no son píxeles: son parcelas, manzanas, caminos. Objetos con sentido.")

---
## 📚 Teoría 3: cada objeto, una fila en una tabla

Para clasificar los objetos necesitamos describirlos con números: sus **estadísticas zonales** — para cada segmento, la media y desviación estándar de cada una de las 12 bandas. Eso convierte la imagen en una **tabla**: una fila por objeto, una columna por característica. Y una tabla ya es territorio conocido: es lo que come cualquier algoritmo de aprendizaje automático.

Como nuestros segmentos están perfectamente alineados al píxel, el cálculo es exacto y rapidísimo con `numpy`:

In [ ]:
import pandas as pd

t0 = time.time()
nseg = int(seg.max()) + 1
flat = seg.ravel()
n_px = np.bincount(flat, minlength=nseg)

tabla = {"segment_id": np.arange(1, nseg), "n_px": n_px[1:]}
for b in range(n_bandas):
    v = img[b].ravel()
    suma  = np.bincount(flat, weights=v,     minlength=nseg)
    suma2 = np.bincount(flat, weights=v * v, minlength=nseg)
    media = np.where(n_px > 0, suma / n_px, 0)
    var   = np.maximum(np.where(n_px > 0, suma2 / n_px, 0) - media**2, 0)
    tabla[f"b{b+1}Mean"]   = media[1:]
    tabla[f"b{b+1}StdDev"] = np.sqrt(var)[1:]

df = pd.DataFrame(tabla)
print(f"Tabla de características en {time.time()-t0:.2f} s: "
      f"{df.shape[0]:,} objetos × {df.shape[1]-2} características espectrales")
df.head()

In [ ]:
# Y de regreso al mapa: poligonizamos los objetos (¡con coordenadas reales!)
# y los pintamos por su reflejo en el infrarrojo cercano (banda 8 ≈ vegetación)
import geopandas as gpd

geoms = ({"properties": {"segment_id": int(v)}, "geometry": g}
         for g, v in features.shapes(seg.astype(np.int32), transform=transform) if v != 0)
gdf = gpd.GeoDataFrame.from_features(geoms, crs=crs).dissolve(by="segment_id", as_index=False)
gdf = gdf.merge(df[["segment_id", "b8Mean"]], on="segment_id")

ax = gdf.plot(column="b8Mean", cmap="RdYlGn", figsize=(7.5, 7.5), linewidth=0, legend=True)
ax.set_title("Objetos coloreados por infrarrojo cercano\n(verde = vegetación vigorosa)")
ax.set_axis_off()
plt.show()
print(f"{len(gdf):,} polígonos georreferenciados — se pueden exportar a GeoPackage y abrir en QGIS")

---
## 🧪 Experimenta tú

La celda de abajo está lista para que juegues con los **dos parámetros clave** del algoritmo. Cambia los valores, ejecútala y observa cómo cambia el mapa:

- `minSegmentSize` — el tamaño mínimo de objeto en píxeles. ¿Qué pasa con 10? ¿Y con 200? *(pista: piensa en qué nivel de detalle territorial necesita tu análisis)*
- `numClusters` — cuántas "familias espectrales" busca el paso 1. ¿Qué pasa con 15? ¿Y con 100?

In [ ]:
# 🧪 Tu laboratorio: cambia estos dos valores y vuelve a ejecutar (Shift+Enter)
MIS_CLUSTERS = 30
MI_TAMANO_MINIMO = 100

t0 = time.time()
mi_res = shepherd_pure.doShepherdSegmentation(
    img, numClusters=MIS_CLUSTERS, minSegmentSize=MI_TAMANO_MINIMO,
    imgNullVal=nodata, fixedKMeansInit=True)
mi_seg = mi_res.segimg

bordes = ndimage.maximum_filter(mi_seg, size=2) != ndimage.minimum_filter(mi_seg, size=2)
vis = rgb.copy(); vis[bordes] = [0, 1, 1]
plt.figure(figsize=(7, 7)); plt.imshow(vis)
plt.title(f"numClusters={MIS_CLUSTERS}, minSegmentSize={MI_TAMANO_MINIMO} → "
          f"{int(mi_seg.max()):,} objetos ({time.time()-t0:.1f} s)")
plt.axis("off"); plt.show()

---
## 🎓 Lo que acabas de lograr

1. Ejecutaste **Python científico completo dentro de tu navegador** — sin instalar nada, sin nube, sin permisos de administrador.
2. Entendiste qué es una **geomediana Sentinel-2** y por qué cada píxel es un vector de 12 mediciones.
3. Aplicaste el algoritmo de **segmentación de Shepherd** — el mismo de uso operativo en agencias de monitoreo — y viste sus 3 pasos por dentro.
4. Convertiste la imagen en una **tabla de objetos con características espectrales**: el insumo exacto para el siguiente paso del curso, la **clasificación con aprendizaje automático** (que también corre aquí, en tu navegador).

En la siguiente sesión: etiquetas de verdad-terreno, entrenamiento de un clasificador y el mapa urbano/no-urbano de Aguascalientes completo.

---
*Implementación de segmentación validada bit a bit contra `pyshepseg` (ubarsc). Datos: geomediana Sentinel-2 2020 (Copernicus / Digital Earth). Plataforma: JupyterLite + Pyodide (WebAssembly). Código y cadena de verificación: [github.com/abxda/portable-satelital](https://github.com/abxda/portable-satelital).*